# Deep Agents 101: Template

This is a blank version of the Deep Agents 101 workshop: same steps (harness, system prompt, tools), no persona or use case baked in. Use it as a starting point to build your own agent.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-openai langgraph tavily-python

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# To use an Anthropic/OpenAI/etc. key instead, add api key to .env

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# To use a different key, replace the block above, for example:
# from langchain.chat_models import init_chat_model
# model = init_chat_model("anthropic:claude-haiku-4-5")

## 1: Invoking The Agent

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

# Write a message here
message = "Hello World"

result = agent.invoke({"messages": [{"role": "user", "content": message}]})
print(result["messages"][-1].content)

## 2: Set its role with a system prompt

In [ ]:
system_prompt = ""

# Examples:
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a research agent. Investigate a topic thoroughly and summarize the key findings with sources."
# system_prompt = "You are a support agent for a cooking app. Only answer questions about cooking, recipes, or the app's features. Politely decline anything else and redirect back to those topics."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

# Replace the ___ placeholder below with your own content
result = agent.invoke({"messages": [{"role": "user", "content":
    "Write three sentences about ___."
}]})
print(result["messages"][-1].content)

## 3: Give it a custom tool

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

The tools we have provided below:

- `roll_dice`: rolls one or more dice (defaults to a single 6-sided die) and returns the results.
- `password_generator`: generates a random password of a given length.
- `random_joke`: fetches a random joke from a public API.
- `currency_convert`: converts an amount between currencies using current exchange rates.
- `random_fact`: fetches a random useless-but-true fact.
- `weather_lookup`: gets the current weather for a city (needs `OPENWEATHER_API_KEY` in `.env`).
- `movie_lookup`: gets the plot, cast, and rating for a movie title (needs `OMDB_API_KEY` in `.env`).
- `stock_quote`: gets the current price for a stock ticker (needs `ALPHAVANTAGE_API_KEY` in `.env`).
- `pokemon_lookup`: looks up a Pokemon's height, weight, types, and abilities by name.
- `random_pokemon`: fetches a random Pokemon's height, weight, types, and abilities.


In [ ]:
from langchain.tools import tool

@tool
def roll_dice(sides: int = 6, count: int = 1) -> str:
    """Roll `count` dice with `sides` sides each."""
    import random
    rolls = [random.randint(1, sides) for _ in range(count)]
    return f"Rolled: {rolls} (total: {sum(rolls)})"

@tool
def password_generator(length: int = 12) -> str:
    """Generate a random password of the given length."""
    import secrets
    import string
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
    return "".join(secrets.choice(alphabet) for _ in range(length))

@tool
def random_joke() -> str:
    """Fetch a random joke."""
    import requests
    joke = requests.get("https://official-joke-api.appspot.com/random_joke").json()
    return f"{joke['setup']} ... {joke['punchline']}"

@tool
def currency_convert(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another using current exchange rates."""
    import requests
    r = requests.get(
        "https://api.frankfurter.app/latest",
        params={"amount": amount, "from": from_currency.upper(), "to": to_currency.upper()},
    ).json()
    rate = r["rates"].get(to_currency.upper())
    if rate is None:
        return f"Couldn't convert {from_currency} to {to_currency}."
    return f"{amount} {from_currency.upper()} = {rate:.2f} {to_currency.upper()}"

@tool
def random_fact() -> str:
    """Fetch a random useless-but-true fact."""
    import requests
    fact = requests.get("https://uselessfacts.jsph.pl/api/v2/facts/random", params={"language": "en"}).json()
    return fact["text"]

@tool
def weather_lookup(city: str) -> str:
    """Look up the current weather for a city."""
    import os
    import requests
    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return "Set OPENWEATHER_API_KEY in your .env to use this tool."
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": api_key, "units": "imperial"},
    ).json()
    if str(r.get("cod")) != "200":
        return f"Couldn't find weather for '{city}'."
    desc = r["weather"][0]["description"]
    temp = r["main"]["temp"]
    return f"{city.title()}: {desc}, {temp}°F"

@tool
def movie_lookup(title: str) -> str:
    """Look up a movie's plot, cast, and rating by title."""
    import os
    import requests
    api_key = os.environ.get("OMDB_API_KEY")
    if not api_key:
        return "Set OMDB_API_KEY in your .env to use this tool."
    r = requests.get("https://www.omdbapi.com/", params={"t": title, "apikey": api_key}).json()
    if r.get("Response") == "False":
        return f"Couldn't find a movie titled '{title}'."
    return f"{r['Title']} ({r['Year']}): {r['Plot']} Starring: {r['Actors']}. IMDb rating: {r['imdbRating']}."

@tool
def stock_quote(symbol: str) -> str:
    """Look up the current price of a stock by ticker symbol."""
    import os
    import requests
    api_key = os.environ.get("ALPHAVANTAGE_API_KEY")
    if not api_key:
        return "Set ALPHAVANTAGE_API_KEY in your .env to use this tool."
    r = requests.get(
        "https://www.alphavantage.co/query",
        params={"function": "GLOBAL_QUOTE", "symbol": symbol.upper(), "apikey": api_key},
    ).json()
    quote = r.get("Global Quote", {})
    price = quote.get("05. price")
    if not price:
        return f"Couldn't find a quote for '{symbol}'."
    return f"{symbol.upper()}: ${float(price):.2f}"

@tool
def pokemon_lookup(name: str) -> str:
    """Look up a Pokemon's height, weight, types, and abilities by name."""
    import requests
    r = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name.lower()}")
    if r.status_code != 200:
        return f"Couldn't find a Pokemon named '{name}'."
    data = r.json()
    types = ", ".join(t["type"]["name"] for t in data["types"])
    abilities = ", ".join(a["ability"]["name"] for a in data["abilities"])
    return (f"{data['name'].title()}: height {data['height']}, weight {data['weight']}, "
            f"type(s) {types}, abilities: {abilities}")

@tool
def random_pokemon() -> str:
    """Fetch a random Pokemon's height, weight, types, and abilities."""
    import random
    import requests
    dex_number = random.randint(1, 1025)
    data = requests.get(f"https://pokeapi.co/api/v2/pokemon/{dex_number}").json()
    types = ", ".join(t["type"]["name"] for t in data["types"])
    abilities = ", ".join(a["ability"]["name"] for a in data["abilities"])
    return (f"{data['name'].title()}: height {data['height']}, weight {data['weight']}, "
            f"type(s) {types}, abilities: {abilities}")

To write your own tool instead of using the examples, add it below.

In [ ]:
# Pick a tool, or write your own @tool function

agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[flip_coin])

# Adjust this prompt to fit whichever tool you picked above
result = agent.invoke({"messages": [{"role": "user", "content":
    "Use your tool to look up Pikachu, then tell me what you found."
}]})
print(result["messages"][-1].content)

## OPTIONAL: Human-in-the-loop, approve/reject a risky action before it happens

**What this covers:** how `interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

**The payoff:** any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

Every tool call passes through an `Interrupt?` check. If a call matches a rule configured in `interrupt_on`, the agent pauses and hands control to a human instead of executing it directly. The human can approve the call as written, edit its arguments before it runs, or reject it outright, and the agent resumes from exactly where it paused.

This pause only works because a `checkpointer` is attached to the agent: it saves the agent's state (its messages, files, and progress so far) at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. Resuming looks like calling `agent.invoke` again with `Command(resume={"decisions": [{"type": "approve"}]})` (or `"edit"` / `"reject"`), rather than starting a new conversation from scratch.

### Try it yourself

Build the pause described above, using the tool you picked in section 3. Gate it with `interrupt_on`, then resume the run with an approval decision.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

checkpointer = InMemorySaver()

# TODO: replace "pokemon_lookup" with the name of the tool you chose in section 3
agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[chosen_tool],
    interrupt_on={"pokemon_lookup": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "hitl-demo"}}

# Reuse (or adjust) the prompt from section 3, it should trigger the same tool call
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "Use your tool to look up Pikachu, then tell me what you found."
    }]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above should pause instead of finishing, because your tool matched `interrupt_on`. Run the cell below to resume it with an approval decision.

In [ ]:
# Other decisions you could pass here instead of "approve":
#   {"type": "reject", "message": "..."}  # skip the tool call; `message` (optional) tells the agent why, so it doesn't just retry
#   {"type": "edit", "edited_action": {"name": "pokemon_lookup", "args": {"name": "charmander"}}}  # run the tool, but with different args than the model chose
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

## Wrap-up

You built: a filesystem-backed agent, a way to swap personas, and a custom tool.

Also covered: human-in-the-loop gating, added with a single argument.

This is a blank template: swap in your own system prompt, starter prompt, and tools to build a completely different agent.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, and deployment.